# 1 - INSTALAMOS DEPENDENCIAS NECESARIAS

In [2]:
!pip install datasets gensim tensorflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.1 MB/s eta 0:00:00


# 2 - IMPORTAMOS LIBRERIAS

In [3]:
import numpy as np
import re
import tensorflow as tf

from datasets import load_dataset
from gensim.models import Word2Vec

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# 3 - CARGA DE DATASET( SOLO USAREMOS 10000 FRASES)

In [4]:
dataset = load_dataset(
    "agentlans/high-quality-english-sentences",
    split="train[:1000]"
)

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.30k [00:00<?, ?B/s]

train.txt.gz:   0%|          | 0.00/85.5M [00:00<?, ?B/s]

test.txt.gz:   0%|          | 0.00/9.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1534699 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/170522 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1000
})

# Convertimos el dataset a una lista de frases

In [6]:
sentences = [str(row["text"]) if "text" in row else str(row) for row in dataset]

sentences[:5]

['Soon we dropped into a living forest, where cold-tolerant evergreens and boreal animals still evoke the Canadian heritage of an ecosystem pushed south by glaciers 20,000 years ago.',
 'Annual population growth rate (2011 est., CIA World Factbook): 1.284%.',
 'This has led to the recent banning of Neonics in the EU, however the US and Canada are still using this chemical pesticide.',
 "In addition, these colors weren't confined to a province but rather irregularly scattered across various regions over all of China.",
 'A family member or a support person may stay with a patient during recovery.']

# Limpieza básica del texto

In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_sentences = [clean_text(sentence) for sentence in sentences]

# Eliminamos frases muy cortas
clean_sentences = [s for s in clean_sentences if len(s.split()) >= 5]

clean_sentences[:5]

['soon we dropped into a living forest where coldtolerant evergreens and boreal animals still evoke the canadian heritage of an ecosystem pushed south by glaciers 20000 years ago',
 'annual population growth rate 2011 est cia world factbook 1284',
 'this has led to the recent banning of neonics in the eu however the us and canada are still using this chemical pesticide',
 'in addition these colors werent confined to a province but rather irregularly scattered across various regions over all of china',
 'a family member or a support person may stay with a patient during recovery']

# Tokenización con Keras

In [8]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(clean_sentences)

total_words = len(tokenizer.word_index) + 1

print("Total de palabras en el vocabulario:", total_words)

Total de palabras en el vocabulario: 6194


In [9]:
input_sequences = []

for sentence in clean_sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(2, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("Cantidad de secuencias:", len(input_sequences))
print(input_sequences[:5])

Cantidad de secuencias: 19566
[[1255, 26, 2174], [1255, 26, 2174, 60], [1255, 26, 2174, 60, 7], [1255, 26, 2174, 60, 7, 369], [1255, 26, 2174, 60, 7, 369, 879]]


In [10]:
# Longitud máxima de las secuencias
max_sequence_len = max([len(seq) for seq in input_sequences])

# Rellenamos con ceros al inicio
input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding="pre"
)

input_sequences[:5]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0, 1255,   26,
        2174],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    

# Separar entradas X y salida y

In [11]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Forma de X: (19566, 88)
Forma de y: (19566,)


In [12]:
# Convertimos y a formato categórico
y = to_categorical(y, num_classes=total_words)

print("Forma de y categórico:", y.shape)

Forma de y categórico: (19566, 6194)


# Entrenar embeddings Word2Vec
Word2Vec aprende vectores de palabras según el contexto.

Luego usaremos esos vectores para inicializar la capa Embedding de TensorFlow.

In [13]:
# Tokenizamos las frases para Word2Vec
tokenized_sentences = [sentence.split() for sentence in clean_sentences]

embedding_dim = 100

word2vec_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=1,
    workers=4
)

print("Word2Vec entrenado correctamente")

Word2Vec entrenado correctamente


In [14]:
# Ejemplo: palabras similares
word2vec_model.wv.most_similar("learning", topn=5)

[('help', 0.9606510996818542),
 ('will', 0.9591044783592224),
 ('if', 0.9590904712677002),
 ('this', 0.9590028524398804),
 ('were', 0.9589509963989258)]

#  Crear matriz de embeddings

In [15]:
embedding_matrix = np.zeros((total_words, embedding_dim))

for word, index in tokenizer.word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[index] = word2vec_model.wv[word]

print("Forma de la matriz de embeddings:", embedding_matrix.shape)

Forma de la matriz de embeddings: (6194, 100)


# Crear modelo RNN con LSTM

In [16]:
model = Sequential()

model.add(
    Embedding(
        input_dim=total_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_sequence_len - 1,
        trainable=True
    )
)

model.add(LSTM(128))
model.add(Dropout(0.3))
model.add(Dense(total_words, activation="softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       619,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 619,400 (2.36 MB)

 Trainable params: 619,400 (2.36 MB)

 Non-trainable params: 0 (0.00 B)

# Entrenar el modelo

In [17]:
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=128,
    verbose=1
)

Epoch 1/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.0558 - loss: 7.5512
Epoch 2/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.0675 - loss: 7.0409
Epoch 3/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0770 - loss: 6.8694
Epoch 4/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0842 - loss: 6.7318
Epoch 5/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0928 - loss: 6.6178


# Función para predecir la siguiente palabra

In [18]:
def predict_next_word(seed_text):
    seed_text = clean_text(seed_text)

    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding="pre"
    )

    predicted_probs = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted_probs)

    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word

    return ""

In [19]:
predict_next_word("machine learning is")

'the'

In [20]:
def complete_sentence(seed_text, next_words=10):
    result = seed_text

    for _ in range(next_words):
        next_word = predict_next_word(result)
        result += " " + next_word

    return result

In [21]:
complete_sentence("machine learning is", next_words=8)

'machine learning is the the the the the the the the'